# 11 - Final inference: all systems on the full test set + the unseen questions (Xt)

> Run with **Save Version -> Save & Run All (Commit)**. Settings: **Accelerator = GPU T4 x2**, **Internet = On**.
> This is the **inference notebook** of the submission. It loads the trained checkpoints (no training happens here) and generates
> answers for **all 1,500 test questions** (our 10% hold-out split, never seen in training) with every system, and for the
> **1,000 unseen questions (Xt)** with the final system. It then scores every system.

**Inputs (Add Input):**
1. `train_processed.csv` (dataset `train-processed1`) and `test_processed.csv` (dataset `train-processed`: the 1,000 unseen questions)
2. `question_neighbours.json` (dataset `nlp-qna-retrieval`: retrieval order)
3. `question_neighbours_reranked.json.gz` (Your Work -> re-ranker notebook 07 output)
4. the Phase 4 checkpoints (dataset `nlp-qna-adapters`: `nn/`, `none/`)
5. the final checkpoint (Your Work -> retrain notebook 08 output: `adapters/nn/final`)
6. *optional, saves about 5 GPU hours:* Your Work -> notebook 09 output. Its answers (300 test questions per system + the Xt answers)
   are reused **only if** the checkpoint file hash, the decoding settings and the exact prompt are identical; anything else is regenerated.
7. *only when resuming:* this notebook's own previous output (see "If it stops early" at the end).

| system | model | reference answers in the prompt |
|---|---|---|
| B0a (original) | none - copy the nearest training answer | - |
| B0a (re-ranked) | none - copy the top re-ranked training answer | - |
| B1 | Qwen2.5-3B-Instruct zero-shot | none |
| B2 | QLoRA adapter v1 `none` (4,000 rows) | none |
| P1 | QLoRA adapter v1 `nn` (4,000 rows) | top-2, original order |
| **P2 (final)** | **QLoRA adapter v2 (all 12,825 rows)** | **top-2, re-ranked** |

Every generated system uses the same decoding (chosen in notebook 06): greedy, repetition penalty 1.1 on the answer's own tokens,
no repeated 12-token n-gram in the answer, at most 1,280 new tokens.
**Xt:** exact repeats of a training question -> the stored medoid answer (router); every other question -> P2.

**Outputs:** `xt_predictions.csv` (submission), `test_predictions.csv` (final system, all 1,500 test questions),
`test_predictions_all_systems.csv`, `metrics.json`, `per_item_scores.csv`, `gen/` (every raw answer), `jobs.jsonl` (every prompt).

In [1]:
!pip install -q -U peft bitsandbytes accelerate huggingface_hub rouge-score sacrebleu
!pip install -q bert-score sentence-transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
%%writefile prompts.py
"""Phase 4: build the training / inference prompts.

Modes
  none : question only                                   (baseline B2: plain fine-tuning)
  nn   : question + the 2 nearest training Q&A pairs      (proposed model without PDFs)
  full : nn + source-page excerpts                         (proposed model with PDFs)

Neighbours for TRAIN questions already exclude the question's own group (see Phase 2), so the model is trained to
synthesise an answer from related-but-different answers, not to copy an identical one.
Pure ASCII source on purpose.
"""
import json
import random

SYSTEM = (
    "You are an expert Indian agricultural advisor. Answer the farmer's question in English with a detailed, "
    "well-structured advisory: key facts or symptoms, thresholds where relevant, cultural, biological and chemical "
    "management with exact doses and units, and safety notes. Use the reference material when it applies to this "
    "question; never invent doses."
)


def trim_words(text, n):
    w = text.split()
    return " ".join(w[:n]) + (" ..." if len(w) > n else "")


def trim_tokens(text, n, tok):
    """Cut text to n tokens. Token budgets (not word counts) keep every prompt inside the length limit: numeric-heavy
    text can use twice as many tokens per word as prose."""
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= n else tok.decode(ids[:n]) + " ..."


def build_user(mode, question, refs, excerpts, ref_words=300, pdf_words=180, tok=None, ref_tokens=380, pdf_tokens=200):
    """refs: list of (question, answer); excerpts: list of (label, text). With tok given, trimming is by tokens."""
    def cut(text, words, tokens):
        return trim_tokens(text, tokens, tok) if tok is not None else trim_words(text, words)
    if mode == "none":
        return "Question: " + question
    parts = ["Reference answers to similar questions:"]
    for i, (q, a) in enumerate(refs, 1):
        parts.append("[%d] Q: %s\nA: %s" % (i, q, cut(a, ref_words, ref_tokens)))
    if mode == "full" and excerpts:
        parts.append("Source excerpts:")
        for label, t in excerpts:
            parts.append("[%s] %s" % (label, cut(t, pdf_words, pdf_tokens)))
    parts.append("Question: " + question)
    return "\n\n".join(parts)


class Context:
    """Holds the lookups needed to build prompts for any query set."""

    def __init__(self, rows, page_text=None, chunk_text=None):
        self.qid2row = {r["question_id"]: r for r in rows}
        self.rows = rows
        self.page_text = page_text or {}      # (doc_id, page) -> first chunk text of that page
        self.chunk_text = chunk_text or []    # chunk id -> text
        self._cit = {}

    def citations(self, qid):
        if qid not in self._cit:
            self._cit[qid] = json.loads(self.qid2row[qid]["citations_json"])
        return self._cit[qid]

    def propagated_pages(self, neighbour_ids, limit=2):
        """Pages cited by the nearest training questions, in neighbour order. Returns [(label, text)]."""
        out, seen = [], set()
        for nid in neighbour_ids:
            for c in self.citations(nid):
                if c.get("kind") != "zoho":
                    continue
                for p in c.get("pages", []):
                    key = (c["doc_id"], p)
                    if key in self.page_text and key not in seen:
                        seen.add(key)
                        name = (c.get("name") or "source")[:60]
                        out.append(("%s, p.%d" % (name, p), self.page_text[key]))
                        if len(out) == limit:
                            return out
        return out


def make_example(ctx, mode, qid, question, neighbours, chunk_hits=None, answer=None, noise=0.0, rng=None,
                 pool_qids=None, n_refs=2, tok=None):
    """neighbours: [[question_id, score], ...] best first."""
    rng = rng or random.Random(0)
    nids = [n[0] for n in neighbours]
    refs_ids = nids[:n_refs]
    if noise and pool_qids and len(refs_ids) == n_refs and rng.random() < noise:
        refs_ids[-1] = rng.choice(pool_qids)      # RAFT-style distractor: the model learns to ignore bad context
    refs = [(ctx.qid2row[i]["question"], ctx.qid2row[i]["final_answer"]) for i in refs_ids]
    excerpts = []
    if mode == "full":
        excerpts = ctx.propagated_pages(nids[:3])
        if len(excerpts) < 2 and chunk_hits and ctx.chunk_text:
            for cid in chunk_hits:
                if len(excerpts) == 2:
                    break
                excerpts.append(("retrieved passage", ctx.chunk_text[cid]))
        if noise and excerpts and pool_qids and rng.random() < noise:
            other = ctx.propagated_pages([rng.choice(pool_qids)], limit=1)
            if other:
                excerpts[-1] = other[0]
    ex = {"qid": qid, "mode": mode, "system": SYSTEM, "user": build_user(mode, question, refs, excerpts, tok=tok)}
    if answer is not None:
        ex["answer"] = answer
    return ex

Writing prompts.py


In [3]:
%%writefile eval_harness.py
"""Single evaluation harness used by every system (baselines, ablations, final model).

    from eval_harness import evaluate
    scores = evaluate(preds, refs)            # dicts {question_id: text}
    scores = evaluate(preds, refs, bertscore=True, meteor=True)   # slower extras

Metrics: ROUGE-1/2/L F1, corpus BLEU (sacrebleu), length ratio, agronomic fact recall/precision/F1
(number+unit pairs, formulation-tagged chemicals such as "dinotefuran 20 SG", binomials), and optionally
METEOR (nltk) and BERTScore-F1. Extras are lazy-imported so the core runs with rouge-score + sacrebleu only.
"""
import re
import statistics as st

from rouge_score import rouge_scorer

_UNIT = {"gm": "g", "gram": "g", "grams": "g", "gms": "g", "litre": "l", "litres": "l", "lit": "l", "liter": "l",
         "liters": "l", "hectare": "ha", "hectares": "ha", "per cent": "%", "percent": "%", "quintals": "q",
         "quintal": "q", "tonnes": "t", "tonne": "t", "day": "days", "kgs": "kg"}
NUM_UNIT_RE = re.compile(
    r"(\d+(?:\.\d+)?)\s*(?:-|\u2013|to)?\s*(\d+(?:\.\d+)?)?\s*"
    r"(kg|kgs|gms?|grams?|g|mg|ml|litres?|liters?|lit|l|quintals?|q|tonnes?|t|ha|hectares?|acre|cm|mm|"
    r"days?|per\s*cent|percent|%|ppm)\b"
    r"(?:\s*(?:/|per)\s*(ha|hectare|acre|litres?|liters?|lit|l|plant|kg|m2|10\s*l))?", re.I)
CHEM_RE = re.compile(r"\b([A-Za-z][A-Za-z\-]{3,})\s+\d+(?:\.\d+)?\s*%?\s*"
                     r"(SG|SL|EC|WP|WG|SC|FS|DP|GR|ZC|OD|EW|CS|ME|WS|DF)\b")
BINOM_RE = re.compile(r"\(([A-Z][a-z]{3,}\s+[a-z]{4,})\)")


def _u(x):
    x = re.sub(r"\s+", " ", x.lower().strip())
    return _UNIT.get(x, x)


def extract_facts(text):
    facts = set()
    for m in NUM_UNIT_RE.finditer(text):
        a, b, unit, per = m.groups()
        facts.add("%s%s%s%s" % (a, "-" + b if b else "", _u(unit), "/" + _u(per) if per else ""))
    for m in CHEM_RE.finditer(text):
        facts.add("chem:" + m.group(1).lower())
    for m in BINOM_RE.finditer(text):
        facts.add("sp:" + m.group(1).lower())
    return facts


def _fact_scores(preds, refs):
    r_list, p_list, f_list = [], [], []
    for p, r in zip(preds, refs):
        fr, fp = extract_facts(r), extract_facts(p)
        if not fr:
            continue
        inter = len(fr & fp)
        rec = inter / len(fr)
        prec = inter / len(fp) if fp else 0.0
        r_list.append(rec)
        p_list.append(prec)
        f_list.append(0.0 if rec + prec == 0 else 2 * rec * prec / (rec + prec))
    n = max(1, len(r_list))
    return {"fact_recall": sum(r_list) / n, "fact_precision": sum(p_list) / n, "fact_f1": sum(f_list) / n,
            "fact_n_refs": len(r_list)}


def evaluate(preds, refs, bertscore=False, meteor=False, bert_model="roberta-large"):
    ids = [i for i in refs if i in preds]
    P = [preds[i] or "" for i in ids]
    R = [refs[i] for i in ids]
    sc = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    agg = {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    for p, r in zip(P, R):
        s = sc.score(r, p)
        for k in agg:
            agg[k] += s[k].fmeasure
    n = max(1, len(ids))
    out = {"n": len(ids), "n_missing": len(refs) - len(ids)}
    out.update({k: v / n for k, v in agg.items()})
    import sacrebleu
    out["bleu"] = sacrebleu.corpus_bleu(P, [R]).score / 100.0
    pl, rl = [len(p.split()) for p in P], [len(r.split()) for r in R]
    out["len_ratio_mean"] = (sum(pl) / n) / max(1e-9, sum(rl) / n)
    out["len_ratio_median_per_item"] = st.median([a / max(1, b) for a, b in zip(pl, rl)]) if ids else 0.0
    out.update(_fact_scores(P, R))
    if meteor:
        from nltk.translate.meteor_score import meteor_score
        out["meteor"] = sum(meteor_score([r.split()], p.split()) for p, r in zip(P, R)) / n
    if bertscore:
        from bert_score import score as bs
        _, _, f1 = bs(P, R, model_type=bert_model, lang="en", batch_size=16, verbose=False)
        out["bertscore_f1"] = float(f1.mean())
    return out


def evaluate_by_group(preds, refs, groups, **kw):
    """groups: {name: set(question_ids)} -> {name: scores}; e.g. novel vs repeated, or per crop."""
    return {g: evaluate(preds, {i: refs[i] for i in ids if i in refs}, **kw) for g, ids in groups.items()}


if __name__ == "__main__":
    t = "Spray dinotefuran 20 SG @ 60 g/acre; seed rate 25 kg/ha. Pest: Amrasca devastans (Amrasca devastans)."
    print(extract_facts(t))
    print(evaluate({"a": t}, {"a": t}))

Writing eval_harness.py


In [4]:
%%writefile infer.py
"""Phase 5 generation worker: one process per GPU.

Reads a jobs file (jsonl: uid, adapter, prio, system, user), generates greedy answers in batches, and appends every
finished batch to --out immediately, so a crashed or time-limited run resumes where it stopped (done uids are skipped).
adapter is "nn" / "none" (a LoRA folder under --adapter_dir) or "base" (the same model with the adapter switched off:
the zero-shot baseline B1).  Pure ASCII source on purpose.

    CUDA_VISIBLE_DEVICES=0 python infer.py --jobs jobs.jsonl --adapter_dir /path --adapters nn --out gen_gpu0.jsonl
"""
import argparse
import json
import os
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, LogitsProcessor, LogitsProcessorList
from peft import PeftModel


def parse():
    p = argparse.ArgumentParser()
    p.add_argument("--jobs", required=True)
    p.add_argument("--adapter_dir", required=True)
    p.add_argument("--adapters", required=True, help="comma list handled by this worker, e.g. none,base")
    p.add_argument("--out", required=True)
    p.add_argument("--model", default="Qwen/Qwen2.5-3B-Instruct")
    p.add_argument("--max_new", type=int, default=768)
    p.add_argument("--batch", type=int, default=12)
    p.add_argument("--rep_penalty", type=float, default=1.05, help="HF repetition penalty (acts on prompt AND answer tokens)")
    p.add_argument("--gen_rep_penalty", type=float, default=1.0, help="penalty on tokens of the ANSWER only (1.0 = off)")
    p.add_argument("--gen_no_repeat", type=int, default=0, help="forbid repeating any n-gram of the ANSWER (0 = off)")
    p.add_argument("--do_sample", action="store_true")
    p.add_argument("--temperature", type=float, default=0.7)
    p.add_argument("--top_p", type=float, default=0.9)
    p.add_argument("--seed", type=int, default=1234)
    p.add_argument("--hours", type=float, default=10.0)
    p.add_argument("--no_quant", action="store_true")
    return p.parse_args()


class GenRepetitionPenalty(LogitsProcessor):
    """Repetition penalty over the generated tokens only (the HF one also penalises every prompt token, which
    discourages copying wording from the reference answers in the prompt). Assumes left padding."""

    def __init__(self, penalty, prompt_len):
        self.p, self.plen = penalty, prompt_len

    def __call__(self, input_ids, scores):
        g = input_ids[:, self.plen:]
        if g.shape[1] == 0:
            return scores
        s = torch.gather(scores, 1, g)
        s = torch.where(s < 0, s * self.p, s / self.p)
        return scores.scatter(1, g, s)


class GenNoRepeatNGram(LogitsProcessor):
    """Forbid any n-gram that already occurs in the generated answer (prompt excluded). Incremental per row."""

    def __init__(self, n, prompt_len):
        self.n, self.plen, self.seen = n, prompt_len, None

    def __call__(self, input_ids, scores):
        B, L = input_ids.shape
        if self.seen is None:
            self.seen = [dict() for _ in range(B)]
        glen = L - self.plen
        if glen < self.n - 1:
            return scores
        rows = input_ids[:, self.plen:].tolist()
        for b in range(B):
            g = rows[b]
            if glen >= self.n:                      # register the newest complete n-gram
                self.seen[b].setdefault(tuple(g[-self.n:-1]), set()).add(g[-1])
            banned = self.seen[b].get(tuple(g[len(g) - self.n + 1:]))
            if banned:
                scores[b, list(banned)] = -float("inf")
        return scores


def load_model(a, names):
    cuda = torch.cuda.is_available()
    if a.no_quant:
        base = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=torch.float16 if cuda else torch.float32)
        if cuda:
            base = base.to("cuda:0")
    else:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True)
        base = AutoModelForCausalLM.from_pretrained(a.model, quantization_config=bnb, torch_dtype=torch.float16,
                                                    device_map={"": 0})
    real = [n for n in names if n != "base"]
    if not real:
        return base.eval(), False
    model = PeftModel.from_pretrained(base, os.path.join(a.adapter_dir, real[0]), adapter_name=real[0])
    for n in real[1:]:
        model.load_adapter(os.path.join(a.adapter_dir, n), adapter_name=n)
    return model.eval(), True


def generate(model, tok, texts, a, device):
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
    plen = enc["input_ids"].shape[1]
    procs = LogitsProcessorList()
    if getattr(a, "gen_rep_penalty", 1.0) != 1.0:
        procs.append(GenRepetitionPenalty(a.gen_rep_penalty, plen))
    if getattr(a, "gen_no_repeat", 0):
        procs.append(GenNoRepeatNGram(a.gen_no_repeat, plen))
    kw = dict(max_new_tokens=a.max_new, repetition_penalty=a.rep_penalty, pad_token_id=tok.pad_token_id,
              logits_processor=procs)
    if getattr(a, "do_sample", False):
        torch.manual_seed(a.seed)
        kw.update(do_sample=True, temperature=a.temperature, top_p=a.top_p, top_k=0)
    else:
        kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    gen = out[:, enc["input_ids"].shape[1]:]
    res = []
    for g in gen:
        ids = g.tolist()
        n = len(ids)
        while n and ids[n - 1] == tok.pad_token_id:
            n -= 1
        res.append((tok.decode(ids[:n], skip_special_tokens=True).strip(), n))
    return res


def main():
    a = parse()
    names = a.adapters.split(",")
    jobs = [json.loads(l) for l in open(a.jobs, encoding="utf-8")]
    jobs = [j for j in jobs if j["adapter"] in names]
    done = set()
    if os.path.exists(a.out):
        done = {json.loads(l)["uid"] for l in open(a.out, encoding="utf-8") if l.strip()}
    todo = [j for j in jobs if j["uid"] not in done]
    print("jobs for %s: %d | already done: %d | to do: %d" % (names, len(jobs), len(done), len(todo)), flush=True)
    if not todo:
        return
    tok = AutoTokenizer.from_pretrained(a.model)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    for j in todo:
        j["text"] = tok.apply_chat_template([{"role": "system", "content": j["system"]}, {"role": "user", "content": j["user"]}],
                                            tokenize=False, add_generation_prompt=True)
        j["plen"] = len(tok(j["text"], add_special_tokens=False)["input_ids"])
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model, has_peft = load_model(a, names)
    t_end = time.time() + a.hours * 3600
    t0 = time.time()
    n_done = 0
    fout = open(a.out, "a", encoding="utf-8")
    for name in names:
        group = sorted((j for j in todo if j["adapter"] == name), key=lambda j: (j["prio"], -j["plen"]))
        if not group:
            continue
        if name != "base" and has_peft:
            model.set_adapter(name)
        print("adapter %s: %d jobs" % (name, len(group)), flush=True)
        i = 0
        bs = a.batch
        while i < len(group):
            if time.time() > t_end:
                print("time limit reached - stopping; rerun to resume", flush=True)
                return
            batch = group[i:i + bs]
            try:
                if name == "base" and has_peft:
                    with model.disable_adapter():
                        res = generate(model, tok, [j["text"] for j in batch], a, device)
                else:
                    res = generate(model, tok, [j["text"] for j in batch], a, device)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs == 1:
                    raise
                bs = max(1, bs // 2)
                print("out of memory - batch size now", bs, flush=True)
                continue
            for j, (text, n) in zip(batch, res):
                fout.write(json.dumps({"uid": j["uid"], "adapter": name, "text": text, "n_tokens": n,
                                       "hit_limit": n >= a.max_new}, ensure_ascii=False) + "\n")
            fout.flush()
            i += len(batch)
            n_done += len(batch)
            el = time.time() - t0
            print("[%5.0f min] %s %d/%d | %.1f s per answer" % (el / 60, name, i, len(group), el / n_done), flush=True)
    print("DONE", n_done, "answers in %.1f min" % ((time.time() - t0) / 60), flush=True)


if __name__ == "__main__":
    main()

Writing infer.py


## 0. Configuration and checkpoints

In [5]:
import glob, os, sys, json, gzip, csv, random, time, shutil, subprocess, hashlib, collections as C
csv.field_size_limit(10 ** 9)
_missing = [f for f in ("prompts.py", "eval_harness.py", "infer.py") if not os.path.exists(f)]
assert not _missing, "missing " + str(_missing) + " - use Run All from the top"
sys.path.insert(0, os.getcwd())
import torch
CPU = os.environ.get("P11_CPU") == "1"     # local smoke-test hook, never set on Kaggle
if not CPU:
    assert torch.cuda.is_available(), "No GPU. Session options -> Accelerator -> GPU T4 x2, then Run All."
N_GPU = torch.cuda.device_count()
print("GPUs:", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

MODEL = os.environ.get("P11_MODEL", "Qwen/Qwen2.5-3B-Instruct")
N_TEST = int(os.environ.get("P11_NTEST", "0"))          # 0 = all 1,500 test questions (smoke tests use fewer)
XT_LIMIT = int(os.environ.get("P11_XT_LIMIT", "0"))     # smoke tests only
DECODE = {"max_new": int(os.environ.get("P11_MAXNEW", "1280")), "rep_penalty": 1.0, "gen_rep_penalty": 1.1, "gen_no_repeat": 12}
BATCH = 10
GEN_HOURS = float(os.environ.get("P11_HOURS", "10.25"))  # workers stop cleanly after this (12 h Kaggle limit incl. scoring)
SPEED = {"base": 6.4, "none_v1": 16.4, "nn_v1": 14.4, "nn_v2": 14.7}   # s/answer measured in notebook 09 (load balancing only)
WORK = os.environ.get("P11_WORK", "/kaggle/working")
INPUT = os.environ.get("P11_INPUT", "/kaggle/input")

def find(pattern, required=True):
    hit = sorted(p.replace("\\", "/") for p in glob.glob(INPUT + "/**/" + pattern, recursive=True))
    if not hit and required:
        raise AssertionError("attach a dataset / notebook output containing " + pattern)
    return hit[0] if hit else None

def op(path):
    return gzip.open(path, "rt", encoding="utf-8") if path.endswith(".gz") else open(path, encoding="utf-8")

def md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""):
            h.update(b)
    return h.hexdigest()

TRAIN_CSV, UNSEEN_CSV = find("train_processed*.csv"), find("test_processed*.csv")
NN_ORIG, NN_RR = find("question_neighbours.json*"), find("question_neighbours_reranked.json*")
V1_NN = os.path.dirname(find("nn/adapter_config.json"))
V1_NONE = os.path.dirname(find("none/adapter_config.json"))
V2_NN = os.path.dirname(find("adapters/nn/final/adapter_config.json"))
V2_SUMMARY = os.path.join(os.path.dirname(V2_NN), "train_summary.json")
V2_STATUS = json.load(open(V2_SUMMARY)) if os.path.exists(V2_SUMMARY) else {}
print("final checkpoint:", V2_NN, "| training status:", V2_STATUS.get("status"), "| steps:", V2_STATUS.get("steps"))

AD = WORK + "/ad"                                  # the three checkpoints under distinct names
for name, srcdir in (("nn_v1", V1_NN), ("none_v1", V1_NONE), ("nn_v2", V2_NN)):
    if not os.path.exists(AD + "/" + name):
        shutil.copytree(srcdir, AD + "/" + name)
AD_HASH = {n: md5(os.path.join(AD, n, "adapter_model.safetensors")) for n in ("nn_v1", "none_v1", "nn_v2")}
json.dump({"model": MODEL, "decoding": DECODE, "adapter_md5": AD_HASH}, open(WORK + "/run_config.json", "w"), indent=1)
print("checkpoint md5:", AD_HASH)
print("decoding:", DECODE)

GPUs: 2 ['Tesla T4', 'Tesla T4']
final checkpoint: /kaggle/input/notebooks/arth99/notebookc8dc821345/adapters/nn/final | training status: completed | steps: 802
checkpoint md5: {'nn_v1': '57cf9313307c40469d752682027cbb24', 'none_v1': 'ee480b22c7b166d0894c54978c6c8159', 'nn_v2': '74037c6506cafd5ab13f450ab104ce7d'}
decoding: {'max_new': 1280, 'rep_penalty': 1.0, 'gen_rep_penalty': 1.1, 'gen_no_repeat': 12}


## 1. Router, test questions and prompts

In [6]:
import prompts as P
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL)
rows = list(csv.DictReader(open(TRAIN_CSV, encoding="utf-8")))
unseen = list(csv.DictReader(open(UNSEEN_CSV, encoding="utf-8")))
NNO, NNR = json.load(op(NN_ORIG)), json.load(op(NN_RR))
ctx = P.Context(rows)
qid2row = ctx.qid2row

# exact-match router: unseen question text == a training question -> that group's medoid answer
medoid = {r["group_id"]: r for r in rows if r["is_medoid"] == "1"}
routed = {u["question_id"]: medoid[u["matched_group_id"]]["final_answer"]
          for u in unseen if u["in_train"] == "1" and u["matched_group_id"] in medoid}
gen_unseen = [u for u in unseen if u["question_id"] not in routed]
if XT_LIMIT:
    gen_unseen = gen_unseen[:XT_LIMIT]
print("unseen: %d | routed by exact match: %d | to generate: %d" % (len(unseen), len(routed), len(gen_unseen)))

all_test = [r for r in rows if r["split"] == "test"]
test_rows = random.Random(42).sample(all_test, N_TEST) if N_TEST else all_test
print("test questions (never seen in training): %d of %d" % (len(test_rows), len(all_test)))

# (name, adapter, prompt mode, neighbour lists)
SYSTEMS = [("B1 zero-shot base model", "base", "none", None),
           ("B2 fine-tuned v1, no retrieval", "none_v1", "none", None),
           ("P1 fine-tuned v1 + retrieval (Phase 4)", "nn_v1", "nn", NNO),
           ("P2 FINAL fine-tuned v2 + re-ranked retrieval", "nn_v2", "nn", NNR)]
FINAL = "P2 FINAL fine-tuned v2 + re-ranked retrieval"

def mk(adapter, prio, tag, qid, question, mode, NNs, split):
    ex = P.make_example(ctx, mode, qid, question, NNs[split].get(qid, []) if mode != "none" else [], tok=tok)
    return {"uid": "%s|%s|%s" % (adapter, tag, qid), "adapter": adapter, "prio": prio, "system": ex["system"], "user": ex["user"]}

jobs = [mk("nn_v2", 0, "xt", u["question_id"], u["question"], "nn", NNR, "xt") for u in gen_unseen]   # Xt first
for r in test_rows:
    for name, ad, mode, NNs in SYSTEMS:
        jobs.append(mk(ad, 1, "hold", r["question_id"], r["question"], mode, NNs, "test"))
os.makedirs(WORK + "/gen", exist_ok=True)
with open(WORK + "/jobs.jsonl", "w", encoding="utf-8") as f:
    for j in jobs:
        f.write(json.dumps(j, ensure_ascii=False) + "\n")
print("jobs:", len(jobs), dict(C.Counter(j["adapter"] for j in jobs)))

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

unseen: 1000 | routed by exact match: 438 | to generate: 562
test questions (never seen in training): 1500 of 1500
jobs: 6562 {'nn_v2': 2062, 'base': 1500, 'none_v1': 1500, 'nn_v1': 1500}


## 2. Reuse verified earlier answers (notebook 09 output / this notebook's previous run)
An earlier answer is reused only if **all** of these hold: same base model and decoding settings, the checkpoint file has the same
MD5 hash, and the prompt is character-for-character identical. Everything else is generated below.

In [7]:
want = {j["uid"]: j for j in jobs}
reused, reuse_log = {}, []
for jf in sorted(glob.glob(INPUT + "/**/jobs.jsonl", recursive=True)):
    folder = os.path.dirname(jf)
    cfg = {}
    for f in ("run_config.json", "metrics.json"):
        if os.path.exists(os.path.join(folder, f)):
            cfg = json.load(open(os.path.join(folder, f)))
            break
    prev_model = cfg.get("model") or cfg.get("config", {}).get("MODEL")
    if cfg.get("decoding") != DECODE or prev_model != MODEL:
        reuse_log.append({"folder": folder, "used": 0, "reason": "different decoding or base model"}); continue
    ok = {n for n in AD_HASH if os.path.exists(os.path.join(folder, "ad", n, "adapter_model.safetensors"))
          and md5(os.path.join(folder, "ad", n, "adapter_model.safetensors")) == AD_HASH[n]}
    ok.add("base")                                     # base model = no adapter; same MODEL checked above
    prev_jobs = {}
    for l in open(jf, encoding="utf-8"):
        if l.strip():
            j = json.loads(l); prev_jobs[j["uid"]] = j
    used = rejected = 0
    for g in sorted(glob.glob(os.path.join(folder, "gen", "*.jsonl"))):
        for l in open(g, encoding="utf-8"):
            if not l.strip():
                continue
            d = json.loads(l); u = d["uid"]
            if u not in want or u in reused:
                continue
            pj = prev_jobs.get(u)
            if (d["adapter"] in ok and pj and pj["system"] == want[u]["system"] and pj["user"] == want[u]["user"]
                    and d["text"].strip()):
                reused[u] = d; used += 1
            else:
                rejected += 1
    reuse_log.append({"folder": folder, "used": used, "rejected": rejected, "verified_checkpoints": sorted(ok)})
with open(WORK + "/gen/reused.jsonl", "w", encoding="utf-8") as f:
    for d in reused.values():
        f.write(json.dumps(d, ensure_ascii=False) + "\n")
for r in reuse_log:
    print("reuse:", r)
print("reused answers: %d of %d jobs %s" % (len(reused), len(jobs), dict(C.Counter(d["adapter"] for d in reused.values()))))

reuse: {'folder': '/kaggle/input/notebooks/arth99/notebookf1c39cfa36', 'used': 1762, 'rejected': 0, 'verified_checkpoints': ['base', 'nn_v1', 'nn_v2', 'none_v1']}
reused answers: 1762 of 6562 jobs {'nn_v2': 862, 'base': 300, 'none_v1': 300, 'nn_v1': 300}


## 3. Generate - jobs balanced over the GPUs (resumable)

In [8]:
if not CPU:
    from huggingface_hub import snapshot_download
    snapshot_download(MODEL)
done_before = set()
for f in glob.glob(WORK + "/gen/gen_gpu*.jsonl"):                 # same-session re-run: keep what is there
    done_before |= {json.loads(l)["uid"] for l in open(f, encoding="utf-8") if l.strip()}
todo = [j for j in jobs if j["uid"] not in reused and j["uid"] not in done_before]
n_workers = 2 if N_GPU >= 2 else 1
loads, assign = [0.0] * n_workers, [[] for _ in range(n_workers)]
for j in sorted(todo, key=lambda j: (j["prio"], -SPEED[j["adapter"]])):     # greedy: each job to the least-loaded GPU
    k = loads.index(min(loads)); assign[k].append(j); loads[k] += SPEED[j["adapter"]]
print("to generate: %d | estimated hours per GPU: %s" % (len(todo), [round(x / 3600, 1) for x in loads]))
procs = []
for k in range(n_workers):
    if not assign[k]:
        continue
    jf = "%s/jobs_gpu%d.jsonl" % (WORK, k)
    with open(jf, "w", encoding="utf-8") as f:
        for j in assign[k]:
            f.write(json.dumps(j, ensure_ascii=False) + "\n")
    names = [n for n in ("nn_v2", "nn_v1", "none_v1", "base") if any(j["adapter"] == n for j in assign[k])]   # nn_v2 first: Xt first
    log = open("%s/gen/worker%d.log" % (WORK, k), "a")
    cmd = [sys.executable, "infer.py", "--jobs", jf, "--adapter_dir", AD, "--adapters", ",".join(names),
           "--out", "%s/gen/gen_gpu%d.jsonl" % (WORK, k), "--model", MODEL, "--batch", str(BATCH), "--hours", str(GEN_HOURS),
           "--max_new", str(DECODE["max_new"]), "--rep_penalty", str(DECODE["rep_penalty"]),
           "--gen_rep_penalty", str(DECODE["gen_rep_penalty"]), "--gen_no_repeat", str(DECODE["gen_no_repeat"])]
    if CPU:
        cmd.append("--no_quant")
    env = dict(os.environ, TOKENIZERS_PARALLELISM="false")
    if not CPU:
        env["CUDA_VISIBLE_DEVICES"] = str(k)
    procs.append((k, subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)))
    print("started worker", k, names, len(assign[k]), "jobs")
t0 = time.time()
while any(p.poll() is None for _, p in procs):
    time.sleep(120 if not CPU else 2)
    for k, p in procs:
        lines = open("%s/gen/worker%d.log" % (WORK, k), errors="ignore").read().splitlines()
        print("[%4.0f min] worker %d: %s" % ((time.time() - t0) / 60, k, lines[-1][:150] if lines else "(starting)"), flush=True)
for k, p in procs:
    print("worker", k, "exit code", p.returncode)
    if p.returncode:
        print(open("%s/gen/worker%d.log" % (WORK, k), errors="ignore").read()[-3000:])

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

to generate: 4800 | estimated hours per GPU: [8.7, 8.7]
started worker 0 ['nn_v2', 'nn_v1', 'none_v1', 'base'] 2400 jobs
started worker 1 ['nn_v2', 'nn_v1', 'none_v1', 'base'] 2400 jobs
[   2 min] worker 0: adapter nn_v2: 600 jobs
[   2 min] worker 1: adapter nn_v2: 600 jobs
[   4 min] worker 0: [    3 min] nn_v2 10/600 | 17.7 s per answer
[   4 min] worker 1: [    3 min] nn_v2 10/600 | 17.9 s per answer
[   6 min] worker 0: [    5 min] nn_v2 20/600 | 15.4 s per answer
[   6 min] worker 1: [    3 min] nn_v2 10/600 | 17.9 s per answer
[   8 min] worker 0: [    5 min] nn_v2 20/600 | 15.4 s per answer
[   8 min] worker 1: [    6 min] nn_v2 20/600 | 17.8 s per answer
[  10 min] worker 0: [    8 min] nn_v2 30/600 | 16.2 s per answer
[  10 min] worker 1: [    8 min] nn_v2 30/600 | 16.1 s per answer
[  12 min] worker 0: [   11 min] nn_v2 40/600 | 16.6 s per answer
[  12 min] worker 1: [   11 min] nn_v2 40/600 | 16.6 s per answer
[  14 min] worker 0: [   11 min] nn_v2 40/600 | 16.6 s per answe

## 4. Collect the answers

In [9]:
import re
gen = {}
for f in glob.glob(WORK + "/gen/*.jsonl"):
    for l in open(f, encoding="utf-8"):
        if l.strip():
            d = json.loads(l); gen[d["uid"]] = d
missing = [j["uid"] for j in jobs if j["uid"] not in gen]
xt_missing = [u for u in missing if "|xt|" in u]
COMPLETE = not missing
print("answers: %d of %d jobs (%d reused, %d generated) | missing: %d (Xt: %d)" % (
    len(jobs) - len(missing), len(jobs), len(reused), len(jobs) - len(missing) - len(reused), len(missing), len(xt_missing)))
print("STATUS:", "COMPLETE" if COMPLETE else "INCOMPLETE - see 'If it stops early' at the end")
WORD = re.compile(r"\S+")
def is_loop(text, n=8, min_rep=3):
    w = [x.lower().strip(".,;:*#-()") for x in WORD.findall(text)]
    c = C.Counter(tuple(w[i:i + n]) for i in range(len(w) - n + 1))
    return bool(c) and max(c.values()) >= min_rep

answers: 6562 of 6562 jobs (1762 reused, 4800 generated) | missing: 0 (Xt: 0)
STATUS: COMPLETE


## 5. Test-set evaluation: lexical metrics

In [10]:
import numpy as np, pandas as pd
import eval_harness as H
def nearest(NNs, split, q):
    nb = NNs[split].get(q)
    return qid2row[nb[0][0]]["final_answer"] if nb else ""

ids = [r["question_id"] for r in test_rows if all(("%s|hold|%s" % (ad, r["question_id"])) in gen for _, ad, _, _ in SYSTEMS)]
refs = {q: qid2row[q]["final_answer"] for q in ids}
B0O, B0R = "B0a copy nearest answer (original order)", "B0a copy nearest answer (re-ranked)"
systems = {B0O: {q: nearest(NNO, "test", q) for q in ids}, B0R: {q: nearest(NNR, "test", q) for q in ids}}
extra = {B0O: {}, B0R: {}}
for name, ad, mode, NNs in SYSTEMS:
    p, e, cap = {}, 0, 0
    for q in ids:
        d = gen["%s|hold|%s" % (ad, q)]
        cap += d["hit_limit"]
        if d["text"].strip():
            p[q] = d["text"]
        else:
            p[q], e = nearest(NNR, "test", q), e + 1     # empty generation -> nearest answer (counted)
    systems[name] = p
    extra[name] = {"empty_replaced": e, "hit_cap": cap}
SYS = list(systems)
res, per, boot = {n: {} for n in SYS}, {}, {}
for n, p in (systems.items() if ids else []):
    res[n] = H.evaluate(p, refs)
    res[n].update(extra[n])
    res[n]["loops_8gram_x3"] = sum(is_loop(p[q]) for q in ids)
    res[n]["median_words"] = float(np.median([len(p[q].split()) for q in ids]))
print("test questions scored: %d of %d%s" % (len(ids), len(test_rows), "" if len(ids) == len(test_rows) else "  <-- PARTIAL"))
if not ids:
    print("no test question has answers from every system yet - scoring is skipped in this session")

test questions scored: 1500 of 1500


## 6. Test-set evaluation: semantic metrics (BERTScore, bge-m3)

In [11]:
if ids:
    from bert_score import BERTScorer
    from sentence_transformers import SentenceTransformer
    from rouge_score import rouge_scorer
    DEV = "cuda" if torch.cuda.is_available() else "cpu"
    R = [refs[q] for q in ids]
    per = {"rougeL": {}, "bertscore_f1": {}, "bertscore_f1_rescaled": {}, "cos_answer_reference": {}}
    rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    for s in SYS:
        per["rougeL"][s] = np.array([rs.score(refs[q], systems[s][q])["rougeL"].fmeasure for q in ids])
    t0 = time.time()
    scorer = BERTScorer(model_type="roberta-large", num_layers=17, lang="en", rescale_with_baseline=True, device=DEV, batch_size=16)
    base_f = float(scorer.baseline_vals[2])
    for s in SYS:
        _, _, F = scorer.score([systems[s][q] for q in ids], R)
        F = F.cpu().numpy().astype(float)
        per["bertscore_f1_rescaled"][s], per["bertscore_f1"][s] = F, F * (1 - base_f) + base_f
    del scorer
    if DEV == "cuda":
        torch.cuda.empty_cache()
    emb = SentenceTransformer("BAAI/bge-m3", device=DEV)
    emb.max_seq_length = 2048
    if DEV == "cuda":
        emb.half()
    enc = lambda texts: emb.encode(texts, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    E_ref = enc(R)
    for s in SYS:
        per["cos_answer_reference"][s] = (enc([systems[s][q] for q in ids]) * E_ref).sum(1).astype(float)
    del emb
    print("semantic scoring done in %.1f min" % ((time.time() - t0) / 60))
    for s in SYS:
        for m in ("bertscore_f1", "bertscore_f1_rescaled", "cos_answer_reference"):
            res[s][m] = float(per[m][s].mean())

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

... (17 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

semantic scoring done in 32.0 min


## 7. Results table and paired bootstraps

In [12]:
if ids:
    cols = ["rougeL", "rouge1", "rouge2", "bleu", "fact_f1", "bertscore_f1", "bertscore_f1_rescaled", "cos_answer_reference",
            "len_ratio_mean", "median_words", "hit_cap", "loops_8gram_x3", "empty_replaced"]
    table = pd.DataFrame({n: {c: res[n].get(c, np.nan) for c in cols} for n in SYS}).T
    pd.set_option("display.width", 300)
    print("N = %d test questions (reference median %d words)" % (len(ids), np.median([len(refs[q].split()) for q in ids])))
    print(table.round(4).to_string())

    n = len(ids)
    idx = np.random.default_rng(0).integers(0, n, (2000, n))
    def diff_ci(m, a, b):
        d = per[m][a] - per[m][b]
        bd = d[idx].mean(1)
        return {"diff": float(d.mean()), "lo": float(np.percentile(bd, 2.5)), "hi": float(np.percentile(bd, 97.5)),
                "a_wins": int((d > 0).sum()), "b_wins": int((d < 0).sum())}
    B1N, B2N, P1N = "B1 zero-shot base model", "B2 fine-tuned v1, no retrieval", "P1 fine-tuned v1 + retrieval (Phase 4)"
    PAIRS = [(FINAL, B0R, "final - copy re-ranked nearest answer"), (FINAL, B0O, "final - copy nearest answer (original order)"),
             (FINAL, P1N, "final - Phase 4 model"), (FINAL, B2N, "final - no retrieval"), (FINAL, B1N, "final - zero-shot"),
             (B0R, B0O, "re-ranked copy - original copy"), (P1N, B0O, "Phase 4 model - original copy")]
    boot = {m: {lab: diff_ci(m, a, b) for a, b, lab in PAIRS} for m in ("rougeL", "bertscore_f1", "cos_answer_reference")}
    for m, d in boot.items():
        print("\n%s  (paired bootstrap, n=%d, 2000 resamples)" % (m, n))
        for lab, r in d.items():
            flag = "  *" if r["lo"] > 0 or r["hi"] < 0 else ""
            print("  %-46s %+.4f  [95%% CI %+.4f, %+.4f]  wins %d vs %d%s" % (lab, r["diff"], r["lo"], r["hi"], r["a_wins"], r["b_wins"], flag))
    print("\n* = the confidence interval excludes 0")

    # consistency check with notebook 09 (same seed-42 sample of 300)
    h300 = [r["question_id"] for r in random.Random(42).sample(all_test, 300)] if len(all_test) >= 300 else []
    m09 = [f for f in glob.glob(INPUT + "/**/metrics.json", recursive=True) if json.load(open(f)).get("holdout_n") == 300]
    if h300 and set(h300) <= set(ids) and m09:
        ref09 = json.load(open(m09[0]))["systems"]
        print("\nsame 300 questions as notebook 09 - ROUGE-L here vs notebook 09:")
        pos = {q: i for i, q in enumerate(ids)}
        for s in SYS:
            if s in ref09:
                v = per["rougeL"][s][[pos[q] for q in h300]].mean()
                print("  %-46s %.4f vs %.4f %s" % (s, v, ref09[s]["rougeL"], "(identical)" if abs(v - ref09[s]["rougeL"]) < 1e-9 else ""))

N = 1500 test questions (reference median 416 words)
                                              rougeL  rouge1  rouge2    bleu  fact_f1  bertscore_f1  bertscore_f1_rescaled  cos_answer_reference  len_ratio_mean  median_words  hit_cap  loops_8gram_x3  empty_replaced
B0a copy nearest answer (original order)      0.2171  0.4088  0.1496  0.1407   0.1684        0.8425                 0.0667                0.7767          1.0140         421.0      NaN           171.0             NaN
B0a copy nearest answer (re-ranked)           0.2322  0.4278  0.1649  0.1577   0.1993        0.8454                 0.0837                0.7882          0.9780         420.0      NaN           149.0             NaN
B1 zero-shot base model                       0.1371  0.3474  0.0605  0.0307   0.0174        0.8259                -0.0318                0.8207          0.8342         414.0      2.0            18.0             0.0
B2 fine-tuned v1, no retrieval                0.1653  0.3386  0.0966  0.0627   0.05

## 8. The router, scored separately
Only repeated questions can be routed, so this is measured on training questions that appear more than once: the stored answer is compared with a *different* human answer to the same question.

In [13]:
groups = C.defaultdict(list)
for r in rows:
    groups[r["group_id"]].append(r)
rep = [g for g in groups.values() if len(g) >= 2]
rr = random.Random(7)
held, p_med, p_rand = {}, {}, {}
for g in rr.sample(rep, min(1500, len(rep))):
    med = next((x for x in g if x["is_medoid"] == "1"), g[0])
    h = rr.choice([x for x in g if x is not med])
    held[h["question_id"]] = h["final_answer"]
    p_med[h["question_id"]] = med["final_answer"]
    p_rand[h["question_id"]] = rr.choice([x for x in g if x is not h])["final_answer"]
router_res, rand_res = H.evaluate(p_med, held), H.evaluate(p_rand, held)
print("repeated-question groups: %d | router (medoid) rougeL %.4f | random other human answer %.4f (conservative)" % (
    len(held), router_res["rougeL"], rand_res["rougeL"]))

repeated-question groups: 501 | router (medoid) rougeL 0.6225 | random other human answer 0.6099 (conservative)


## 9. Write the prediction files

In [14]:
sfx = "" if COMPLETE else "_PARTIAL"
def write(path, header, items, rowf):
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(header)
        for it in items:
            w.writerow(rowf(it))

# unseen set (Xt): router, else the final model
final, source, baseline = {}, {}, {}
for u in unseen:
    q = u["question_id"]
    if q in routed:
        final[q] = baseline[q] = routed[q]; source[q] = "router"
        continue
    baseline[q] = nearest(NNR, "xt", q)
    d = gen.get("nn_v2|xt|%s" % q)
    if d and d["text"].strip():
        final[q], source[q] = d["text"], "generated"
    else:
        final[q], source[q] = baseline[q], "fallback_nearest"
xt_ok = not xt_missing and len(final) == len(unseen) == 1000 and all(v.strip() for v in final.values())
xs = "" if xt_ok else "_PARTIAL"
write(WORK + "/xt_predictions%s.csv" % xs, ["question_id", "question", "final_answer"], unseen,
      lambda u: [u["question_id"], u["question"], final[u["question_id"]]])
write(WORK + "/xt_predictions_detailed%s.csv" % xs, ["question_id", "source", "final_answer"], unseen,
      lambda u: [u["question_id"], source[u["question_id"]], final[u["question_id"]]])
write(WORK + "/xt_predictions_with_baseline%s.csv" % xs, ["question_id", "question", "final_answer", "baseline_answer"], unseen,
      lambda u: [u["question_id"], u["question"], final[u["question_id"]], baseline[u["question_id"]]])
print("Xt:", "complete" if xt_ok else "INCOMPLETE", "|", dict(C.Counter(source.values())))

# test set: the final system for every test question, and all systems side by side
scored = [qid2row[q] for q in ids]
write(WORK + "/test_predictions%s.csv" % sfx, ["question_id", "question", "reference_answer", "final_answer"], scored,
      lambda r: [r["question_id"], r["question"], r["final_answer"], systems[FINAL][r["question_id"]]])
write(WORK + "/test_predictions_all_systems%s.csv" % sfx, ["question_id", "question", "reference_answer"] + SYS, scored,
      lambda r: [r["question_id"], r["question"], r["final_answer"]] + [systems[s][r["question_id"]] for s in SYS])
flat = pd.DataFrame({"question_id": ids})
for m in per:
    for s in SYS:
        flat["%s|%s" % (m, s)] = per[m][s]
flat.to_csv(WORK + "/per_item_scores%s.csv" % sfx, index=False)
xt_gen = [gen[u] for u in gen if u.startswith("nn_v2|xt|")]
json.dump({"status": "complete" if COMPLETE else "incomplete", "test_n": len(test_rows), "test_scored": len(ids),
           "decoding": DECODE, "model": MODEL, "adapter_md5": AD_HASH, "reuse": reuse_log, "n_reused": len(reused),
           "systems": res, "bootstrap": boot, "router": router_res, "router_random_other": rand_res,
           "routed_share": len(routed) / len(unseen),
           "unseen": {"sources": dict(C.Counter(source.values())), "generated_hit_cap": sum(d["hit_limit"] for d in xt_gen),
                      "generated_loops_8gram_x3": sum(is_loop(d["text"]) for d in xt_gen)},
           "final_checkpoint_training": {k: v for k, v in V2_STATUS.items() if k not in ("loss_log", "eval_log")}},
          open(WORK + "/metrics%s.json" % sfx, "w"), indent=1, default=float)
print("wrote test_predictions%s.csv (%d rows), test_predictions_all_systems%s.csv, per_item_scores%s.csv, metrics%s.json" % (sfx, len(ids), sfx, sfx, sfx))
print("\nSTATUS:", "COMPLETE - all files final" if COMPLETE else "INCOMPLETE - files are marked _PARTIAL; see below")

Xt: complete | {'router': 438, 'generated': 562}
wrote test_predictions.csv (1500 rows), test_predictions_all_systems.csv, per_item_scores.csv, metrics.json

STATUS: COMPLETE - all files final


## 10. A few answers side by side

In [15]:
for q in ids[:3]:
    print("=" * 100, "\nQ:", qid2row[q]["question"][:200])
    print("--- reference :", refs[q][:400].replace("\n", " "))
    print("--- final (P2):", systems[FINAL][q][:400].replace("\n", " "))
    print("--- B0a re-ranked copy:", systems[B0R][q][:250].replace("\n", " "))

Q: What is the recommended fertilizer dose for J&K Apple?
--- reference : A healthy apple orchard needs balanced nutrition for good tree growth, flowering, fruit development and yield. However, the fertilizer requirement is not the same for every orchard. It depends on the age of the tree and the nutrient status of the soil and plant. Therefore, farmers should consider soil and leaf analysis before preparing a fertilizer programme, so that nutrients are applied accordin
--- final (P2): The recommended fertilizer dose for J&K apple varies based on soil type, soil fertility, and the specific variety being grown. However, a general guideline for nutrient application can be provided to ensure optimal growth and yield.  Soil Testing : Before fertilizer application, soil testing is essential to determine nutrient status and to apply balanced and need-based fertilizers.   Nutrient Mana
--- B0a re-ranked copy: The recommended fertilizer dose for pearl millet varies based on soil type, soil fer

## If it stops early
Generation stops cleanly after 10.25 hours so the scoring still fits in Kaggle's 12-hour limit. If the status says **INCOMPLETE**
(files end in `_PARTIAL`): open a new version of this notebook, **Add Input -> Your Work -> this notebook's output** (keep all the
other inputs), and **Save & Run All** again. The previous answers are verified and reused, and only the missing ones are generated.

## After it finishes
Send me the notebook link (or the downloaded output). Output tab: `xt_predictions.csv` (submission), `test_predictions.csv`,
`test_predictions_all_systems.csv`, `metrics.json`, `per_item_scores.csv`, `gen/`, `jobs.jsonl`.